# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [3]:
import pandas as pd
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

In [5]:
# 1. Create a new DataFrame that only includes customers who:
# have a low total_claim_amount (e.g., below $1,000),
# have a response "Yes" to the last marketing campaign.
df.head()


,Unnamed: 0,Customer,State,Customer Lifetime Value,Response,Coverage,Education,Effective To Date,EmploymentStatus,Gender,...,Number of Open Complaints,Number of Policies,Policy Type,Policy,Renew Offer Type,Sales Channel,Total Claim Amount,Vehicle Class,Vehicle Size,Vehicle Type
0,0,DK49336,Arizona,4809.216960,No,Basic,College,2/18/11,Employed,M,...,0.0,9,Corporate Auto,Corporate L3,Offer3,Agent,292.800000,Four-Door Car,Medsize,NaN
1,1,KX64629,California,2228.525238,No,Basic,College,1/18/11,Unemployed,F,...,0.0,1,Personal Auto,Personal L3,Offer4,Call Center,744.924331,Four-Door Car,Medsize,NaN
2,2,LZ68649,Washington,14947.917300,No,Basic,Bachelor,2/10/11,Employed,M,...,0.0,2,Personal Auto,Personal L3,Offer3,Call Center,480.000000,SUV,Medsize,A
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
4,4,QA50777,Oregon,9025.067525,No,Premium,Bachelor,1/17/11,Medical Leave,F,...,NaN,7,Personal Auto,Personal L2,Offer1,Branch,707.925645,Four-Door Car,Medsize,NaN


In [7]:
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)


In [12]:
low_claim_yes = df.loc[
    (df["total_claim_amount"] < 1000) &
    (df["response"] == "Yes")
].copy()

In [14]:
low_claim_yes.head()


,unnamed:_0,customer,state,customer_lifetime_value,response,coverage,education,effective_to_date,employmentstatus,gender,...,number_of_open_complaints,number_of_policies,policy_type,policy,renew_offer_type,sales_channel,total_claim_amount,vehicle_class,vehicle_size,vehicle_type
3,3,XL78013,Oregon,22332.439460,Yes,Extended,College,1/11/11,Employed,M,...,0.0,2,Corporate Auto,Corporate L3,Offer2,Branch,484.013411,Four-Door Car,Medsize,A
8,8,FM55990,California,5989.773931,Yes,Premium,College,1/19/11,Employed,M,...,0.0,1,Personal Auto,Personal L1,Offer2,Branch,739.200000,Sports Car,Medsize,NaN
15,15,CW49887,California,4626.801093,Yes,Basic,Master,1/16/11,Employed,F,...,0.0,1,Special Auto,Special L1,Offer2,Branch,547.200000,SUV,Medsize,NaN
19,19,NJ54277,California,3746.751625,Yes,Extended,College,2/26/11,Employed,F,...,1.0,1,Personal Auto,Personal L2,Offer2,Call Center,19.575683,Two-Door Car,Large,A
27,27,MQ68407,Oregon,4376.363592,Yes,Premium,Bachelor,2/28/11,Employed,F,...,0.0,1,Personal Auto,Personal L3,Offer2,Agent,60.036683,Four-Door Car,Medsize,NaN


In [16]:
low_claim_yes.shape[0]

1399

In [18]:
#Display only the most relevant columns:
low_claim_yes[
    ["customer", "response", "total_claim_amount"]
].head(10)

,customer,response,total_claim_amount
3,XL78013,Yes,484.013411
8,FM55990,Yes,739.200000
15,CW49887,Yes,547.200000
19,NJ54277,Yes,19.575683
27,MQ68407,Yes,60.036683
65,LO57874,Yes,302.400000
67,KR35099,Yes,231.201886
69,QG27547,Yes,374.400000
88,CJ51511,Yes,571.200000
102,VG56765,Yes,114.273025


In [20]:
# 2. Using the original Dataframe, analyze:
# the average monthly_premium and/or customer lifetime value by policy_type and gender for customers who responded "Yes", and
# compare these insights to total_claim_amount patterns, and discuss which segments appear most profitable or low-risk for the company.


yes_customers = df.loc[
    df["response"] == "Yes"
].copy()

In [22]:
segment_analysis = (
    yes_customers
    .groupby(["policy_type", "gender"])
    .agg(
        customers=("customer", "count"),
        average_monthly_premium=("monthly_premium_auto", "mean"),
        average_clv=("customer_lifetime_value", "mean"),
        average_claim=("total_claim_amount", "mean")
    )
    .round(2)
)

In [24]:
segment_analysis

customers  average_monthly_premium  average_clv  \
policy_type    gender                                                    
Corporate Auto F             169                    94.30      7712.63   
               M             154                    92.19      7944.47   
Personal Auto  F             540                    99.00      8339.79   
               M             536                    91.09      7448.38   
Special Auto   F              35                    92.31      7691.58   
               M              32                    86.34      8247.09   

                       average_claim  
policy_type    gender                 
Corporate Auto F              433.74  
               M              408.58  
Personal Auto  F              452.97  
               M              457.01  
Special Auto   F              453.28  
               M              429.53

The highest ratios are approximately
1. Personal Auto women have the highest average CLV at $8,339.79 and form the largest responding segment.
2. However, their average claim is also relatively high at $452.97.
3. Corporate Auto men have the lowest average claim at $408.58 and the highest CLV-to-claim ratio.
4. Corporate Auto men may therefore represent a comparatively valuable and lower-risk segment.
5. Special Auto men also show a strong CLV-to-claim ratio, but this group contains only 32 customers, so the finding is based on a small sample.
6. Personal Auto men have the lowest CLV-to-claim ratio and the highest average claim.

In [26]:
# Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

state_customers = (
    df.groupby("state")
    .size()
    .reset_index(name="customer_count")
)

In [28]:
state_customers = state_customers.sort_values(
    "customer_count",
    ascending=False
)

state_customers

,state,customer_count
1,California,3552
3,Oregon,2909
0,Arizona,1937
2,Nevada,993
4,Washington,888


In [30]:
states_over_500 = state_customers.loc[
    state_customers["customer_count"] > 500
]

states_over_500

,state,customer_count
1,California,3552
3,Oregon,2909
0,Arizona,1937
2,Nevada,993
4,Washington,888


In [32]:
#4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

clv_by_education_gender = (
    df.groupby(["education", "gender"])
    ["customer_lifetime_value"]
    .agg(["max", "min", "median"])
    .round(2)
)

clv_by_education_gender

max      min   median
education            gender                            
Bachelor             F       73225.96  1904.00  5640.51
                     M       67907.27  1898.01  5548.03
College              F       61850.19  1898.68  5623.61
                     M       61134.68  1918.12  6005.85
Doctor               F       44856.11  2395.57  5332.46
                     M       32677.34  2267.60  5577.67
High School or Below F       55277.45  2144.92  6039.55
                     M       83325.38  1940.98  6286.73
Master               F       51016.07  2417.78  5729.86
                     M       50568.26  2272.31  5579.10

Conclusions
- High School or Below men have the highest maximum CLV: $83,325.38.
- High School or Below men also have the highest median CLV: $6,286.73.
- Women with Doctoral education have the lowest median CLV: $5,332.46.
- Bachelor women have a very high maximum CLV of $73,225.96, but their median is only $5,640.51. This suggests that a small number of customers have exceptionally high CLV.